# Add Captions to Images

## Initial Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [19]:
# Initial imports
import json

from collections import Counter
from typing import Dict, Any, List

from tqdm.auto import tqdm

## Training and Using a Transformer to Add Captions

### Preparing Data and the Causal Attention Mask

### Creating and Training a Transformer

## Preparing the Training Dataset

### Downloading and Visualizing Flickr 8k Images

In [15]:
# Total disk size of images
!du -hs /llm_app/learning/build_a_text_to_image_generator_from_scratch/files/Images

1.1G	/llm_app/learning/build_a_text_to_image_generator_from_scratch/files/Images


In [ ]:
# Information about the captions and corresponding tokens for each caption
with open("/llm_app/learning/build_a_text_to_image_generator_from_scratch/files/dataset_flickr8k.json", "r") as fb:
    data: Dict[str, Any] = json.load(fb)

In [ ]:
# Listing 4.1 Splitting data into train and test subsets

train_image_paths: List[str] = []
train_image_captions: List[str] = []
test_image_paths: List[str] = []
test_image_captions: List[str] = []

# How oftem each token appears in the captions
word_freq: Counter = Counter()

# Maximum length of a caption
max_len: int = 50

# Loops through all images in the dataset
for img in tqdm(data["images"], total=len(data["images"]), desc="Processing images into train and test subsets"):
    
    # Miight have more than one caption per image
    captions: List[List[str]] = []
    
    # Gathers all captions for the current image
    for c in img["sentences"]:
        
        # Gathers information about text tokens to create a vocabulary
        word_freq.update(c["tokens"])
        
        if len(c["tokens"]) <= max_len:        
            captions.append(c["tokens"])
    
    if len(captions) == 0:
        continue
    
    # Defines the path to the image
    path: str = "/llm_app/learning/build_a_text_to_image_generator_from_scratch/files/Images/"+img["filename"] 
    
    # Train subset
    if img["split"] in {"train", "val", "restval"}:
        train_image_paths.append(path)
        train_image_captions.append(captions)
    
    # Test subset
    elif img["split"] in {"test"}:
        test_image_paths.append(path)
        test_image_captions.append(captions)

Processing images into train and test subsets:   0%|          | 0/8000 [00:00<?, ?it/s]

In [30]:
print(f">>> Statistics for the training and test subsets")
print(f"\t* number of images in training subset: {len(train_image_paths)}")
print(f"\t* number captions in training subset: {sum([len(c) for c in train_image_captions])}")
print(f"\t* number of images in test subset: {len(test_image_paths)}")
print(f"\t* number captions in test subset: {sum([len(c) for c in test_image_captions])}")

>>> Statistics for the training and test subsets
	* number of images in training subset: 7000
	* number captions in training subset: 35000
	* number of images in test subset: 1000
	* number captions in test subset: 5000


In [31]:
# Make sure that the number of images matches the number of groups of captions in both 
# the training and test sets (recall that each image has five captions)
assert len(train_image_paths) == len(train_image_captions)
assert len(test_image_paths) == len(test_image_captions)

### Building a Vocabulary of Tokens